In [ ]:
import pandas as pd
import numpy as np
import commons as c
import plotly.express as px

# Get datasets

In [ ]:
csv_normal_path = 'results/dataframes/results_normal.csv'
df_normal = pd.read_csv(csv_normal_path, dtype=c.type_dict)

csv_equiv_path = 'results/dataframes/results_equiv.csv'
df_equiv = pd.read_csv(csv_equiv_path, dtype=c.type_dict)

In [ ]:
df = pd.concat([df_equiv, df_normal], ignore_index=True)
df['true_label'] = np.where(df['true_label'] == True, "non-equivalent", "equivalent")
df['predicted_label'] = np.where(df['predicted_label'] == True, "non-equivalent", "equivalent")

# Get Box Plots

In [ ]:
def print_box_plot(df, cat, output_folder, file_name):

    df = df.copy()  # Ensure it's a copy
    cat_range = sorted(df[cat].unique())  # Extract unique values from the column    
    cat_mapping = {category: idx for idx, category in enumerate(cat_range)}
    df['cat_numeric'] = df[cat].map(cat_mapping)  # Map categories to numeric values

    # Add an offset to the x-axis values based on 'true_label' to spread the points
    label_offsets = {
        "equivalent": -0.1,   # Adjust this value as needed
        "non-equivalent": 0.1  # Adjust this value as needed
    }
    
    # Apply the offset to a new column for the x-axis
    df['x_offset'] = df['cat_numeric'] + df['true_label'].map(label_offsets)

    label_mapping = {
        "equivalent": "Equivalent mutant",
        "non-equivalent": "Non-Equivalent mutant"
    }    
    df['true_label'] = df['true_label'].map(label_mapping)
    
    # Create the box plot 
    fig = px.box(
        df, 
        y="noisy_distance", 
        x="x_offset", 
        color="true_label", 
        category_orders={cat: cat_range},
        points=False
    ) 
    
    # Adjust layout for better visualization
    fig.update_layout(
        scattermode="group",
        #scattergap=0.75,
        xaxis=dict(
            title=cat, 
            categoryorder="array", 
            categoryarray=cat_range,
            tickvals=list(range(len(cat_range))),
            ticktext=cat_range
        ),
        yaxis_title="Distance between original and mutant",
        xaxis_title="Characteristic",
        legend_title_text="Expected value",
        boxgroupgap=0
        #boxgap=0
    )
    
    # Save the figure
    c.setup_layout_and_save(fig, "RQ3", output_folder, file_name, yaxis_range=[0, 1])


In [ ]:
columns = ['Qubits_number', 'gates', 'depth', 'singlequbit_gates', 'multiqubit_gates'] 
hardware = ["kyiv", "brisbane", "sherbrooke"]

for m in c.metrics:
    for threshold in c.thresholds:
        df_metric = df[df['metric'] == m]
        df_threshold = df_metric[df_metric['threshold'] == threshold]
        for hw in c.hardware:
            selected_columns = df_threshold[['hardware', 'true_label', 'ideal_distance', 'noisy_distance']]
            file_name = f'{m}_{threshold}_{hw}'
            output_folder = f'results/RQ3/{m}_{threshold}'
            print_box_plot(selected_columns, 'hardware', output_folder, file_name)
        